# Phase 3 — Lean Candidate Generation & Blocking Analysis
### Amazon ML Challenge: Multi-Source Entity Resolution

**Goal:** Maximize $\frac{\text{Recall Ceiling}}{\text{Candidate Set Size}}$ using a high-precision, 5-block combined strategy with cascading stopword pruning and provenance pre-ranking. Candidate sets are kept lean (~15–25 candidates/entity, ~2–3 GB total), adhering to the competition's explicit ranking criterion.

**Sources:**
- **S1 (Query Anchor — strictly preserved):** `train_source1.tsv` / `test_source1.tsv`
- **S2 (Indexed):** `train_source2.tsv` / `test_source2.tsv`
- **S3 (Indexed):** `train_source3.tsv` / `test_source3.tsv`

**5 Combined Blocks:**
1. `exact_name`: Clean base name (legal suffixes stripped)
2. `exact_address`: Clean full address (len > 10)
3. `postal_brand`: Postal code + primary brand token
4. `country_brand`: Country + primary/secondary brand tokens
5. `region_brand`: Country + region + primary brand token
6. `postal_prefix_brand`: 3-digit postal zone + primary brand token

In [2]:
from __future__ import annotations

import csv
import re
import sys
import time
import unicodedata
from array import array
from collections import defaultdict, Counter
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple

# Auto-detect project root
if Path("/content/AMAZON_ML_CHALLENGE-main").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE-main")
elif Path("/content/AMAZON_ML_CHALLENGE").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE")
elif Path("..").resolve().name == "AMAZON_ML_CHALLENGE":
    ROOT = Path("..").resolve()
else:
    ROOT = Path(".").resolve()

DATA = ROOT / "student_resource" / "dataset"
OUTPUT_DIR = ROOT / "data" / "processed" / "phase3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_BLOCK_SIZE = 150
TOP_K_CANDIDATES = 30

GENERIC_TOKENS = {
    "the", "national", "american", "sri", "global", "general", "royal",
    "super", "new", "united", "first", "city", "apex", "shree", "om",
    "india", "us", "star", "all", "best", "prime", "central", "standard",
    "and", "of", "in"
}
LEGAL_SUFFIXES = {
    "pvt", "ltd", "private", "limited", "inc", "incorporated",
    "corp", "corporation", "llc", "llp", "gmbh", "sa", "sarl", "co", "company"
}

print(f"Project Root: {ROOT}")
print(f"Dataset Path: {DATA} (Exists: {DATA.exists()})")
print(f"Phase 3 Output Directory: {OUTPUT_DIR}")

## Step 1: Text Normalization and Cascading Token Extractors

In [4]:
def normalize_text(value: str) -> str:
    value = str(value or "").strip().lower()
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
    return re.sub(r"\s+", " ", value).strip()

def get_clean_tokens(name: str) -> list[str]:
    """Cascading generic token pruner: skips all consecutive generic words."""
    raw_tokens = normalize_text(name).split()
    sig_tokens = [t for t in raw_tokens if t not in GENERIC_TOKENS and t not in LEGAL_SUFFIXES]
    return sig_tokens if sig_tokens else [t for t in raw_tokens if t not in LEGAL_SUFFIXES] or raw_tokens

def postal_tokens(address: str):
    return re.findall(r"\b[a-z0-9][a-z0-9 -]{2,9}\b", address.lower())

def get_region(address: str) -> str:
    parts = [p.strip() for p in address.split(",") if p.strip()]
    return parts[-2] if len(parts) >= 2 else ""

## Step 2: Combined Block Key Generator

In [6]:
def get_keys(row: dict) -> dict:
    raw_addr = row.get("business_address", "")
    raw_name = row.get("business_name_normalized") or row.get("business_name") or ""
    
    clean_toks = get_clean_tokens(raw_name)
    base_name = " ".join(clean_toks)
    
    brand_t1 = clean_toks[0] if len(clean_toks) >= 1 else ""
    brand_t2 = clean_toks[1] if len(clean_toks) >= 2 else ""
    brand_t12 = f"{brand_t1}_{brand_t2}" if (brand_t1 and brand_t2) else brand_t1
    
    address = normalize_text(raw_addr)
    country = normalize_text(row.get("country_normalized") or row.get("country") or "")
    
    postal = ""
    for tok in postal_tokens(raw_addr):
        if any(ch.isdigit() for ch in tok):
            postal = tok
            break
            
    prefix = re.sub(r"[^a-z0-9]", "", postal)[:3] if postal else ""
    region = normalize_text(get_region(raw_addr))
    
    return {
        "exact_name": base_name if base_name else "",
        "exact_address": address if len(address) > 10 else "",
        "postal_brand": f"{postal}|{brand_t1}" if (postal and brand_t1) else "",
        "country_brand": f"{country}|{brand_t12}" if (country and brand_t12 and len(brand_t1) >= 3) else "",
        "region_brand": f"{country}|{region}|{brand_t1}" if (country and region and brand_t1) else "",
        "postal_prefix_brand": f"{prefix}|{brand_t1}" if (prefix and brand_t1) else "",
    }

TIGHT_BLOCKS = [
    "exact_name",
    "exact_address",
    "postal_brand",
    "country_brand",
    "region_brand",
    "postal_prefix_brand",
]

## Step 3: Fast Inverted Indexing over S2 and S3

In [8]:
def eid_to_int(eid: str) -> int:
    try:
        dash = eid.find("-")
        num = int(eid[dash+1:])
        return num if eid.startswith("S2") else -num
    except Exception:
        return 0

def int_to_eid(num: int) -> str:
    return f"S2-{num}" if num > 0 else f"S3-{-num}"

def make_array():
    return array("i")

def build_indexes(s2_path: Path, s3_path: Path, max_rows: int = None):
    indexes = {b: defaultdict(make_array) for b in TIGHT_BLOCKS}
    for s_label, s_file in [("Source 2", s2_path), ("Source 3", s3_path)]:
        print(f"Indexing {s_label} ({s_file.name})...")
        count = 0
        with open(s_file, "r", encoding="utf-8", errors="replace", newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                count += 1
                int_id = eid_to_int(row["entity_id"])
                if int_id == 0:
                    continue
                keys = get_keys(row)
                for b in TIGHT_BLOCKS:
                    k = keys.get(b)
                    if k:
                        bucket = indexes[b][k]
                        if len(bucket) < MAX_BLOCK_SIZE:
                            bucket.append(int_id)
                if max_rows and count >= max_rows:
                    break
        print(f"  --> Completed {s_label}: {count:,} records.")
    return indexes

## Step 4: Full-Scale Execution Pipeline
Executes candidate generation with **Provenance Pre-Ranking** (Top 30 cap per S1 entity).

In [10]:
TRAIN_S1 = DATA / "train" / "train_source1.tsv"
TRAIN_S2 = DATA / "train" / "train_source2.tsv"
TRAIN_S3 = DATA / "train" / "train_source3.tsv"
OUTPUT_CANDIDATES = OUTPUT_DIR / "candidate_pairs.tsv"

print("=== Starting Lean Candidate Generation Pipeline ===")
# Pipeline ready for execution via run_lean_phase3.py